# 14 — Object vs Checker Comparison (Threshold Sweep)

**Purpose:** For each of 9 epsilon × consistency combinations, compare universal circuit masks with token checker masks. Output one CSV per combination plus a summary.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np, pandas as pd, h5py

EPSILONS = [0.001, 0.1, 0.5]
CONSISTENCIES = [0.2, 0.5, 0.8]
N_LAYERS = 8

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

print(f"Data dir: {DATA_DIR}")
print(f"Combinations: {len(EPSILONS)} x {len(CONSISTENCIES)} = {len(EPSILONS) * len(CONSISTENCIES)}")

In [ ]:
# Cell 2 – Keyword map
KEYWORD_MAP = {
    "ast__Import":       {"keyword": "import",   "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__ImportFrom":   {"keyword": "from",     "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__Break":        {"keyword": "break",    "forbidden_nodes": ["Break"]},
    "ast__Pass":         {"keyword": "pass",     "forbidden_nodes": ["Pass"]},
    "ast__Continue":     {"keyword": "continue", "forbidden_nodes": ["Continue"]},
    "ast__Assert":       {"keyword": "assert",   "forbidden_nodes": ["Assert"]},
    "ast__If":           {"keyword": "if",       "forbidden_nodes": ["If", "IfExp"]},
    "ast__While":        {"keyword": "while",    "forbidden_nodes": ["While"]},
    "ast__For":          {"keyword": "for",      "forbidden_nodes": ["For", "AsyncFor", "ListComp", "SetComp", "DictComp", "GeneratorExp"]},
    "ast__Return":       {"keyword": "return",   "forbidden_nodes": ["Return"]},
    "ast__With":         {"keyword": "with",     "forbidden_nodes": ["With", "AsyncWith"]},
    "ast__Raise":        {"keyword": "raise",    "forbidden_nodes": ["Raise"]},
    "ast__Delete":       {"keyword": "del",      "forbidden_nodes": ["Delete"]},
    "ast__Global":       {"keyword": "global",   "forbidden_nodes": ["Global"]},
    "ast__Nonlocal":     {"keyword": "nonlocal", "forbidden_nodes": ["Nonlocal"]},
    "ast__Lambda":       {"keyword": "lambda",   "forbidden_nodes": ["Lambda"]},
    "ast__Yield":        {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__YieldFrom":    {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__Try":          {"keyword": "try",      "forbidden_nodes": ["Try"]},
    "ast__ClassDef":     {"keyword": "class",    "forbidden_nodes": ["ClassDef"]},
    "ast__FunctionDef":  {"keyword": "def",      "forbidden_nodes": ["FunctionDef", "AsyncFunctionDef"]},
    "ast__AsyncFor":     {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncFunctionDef": {"keyword": "async", "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncWith":    {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "builtin__list":     {"keyword": "list",     "forbidden_nodes": [],  "forbidden_names": ["list"]},
    "builtin__set":      {"keyword": "set",      "forbidden_nodes": [],  "forbidden_names": ["set"]},
    "builtin__map":      {"keyword": "map",      "forbidden_nodes": [],  "forbidden_names": ["map"]},
    "builtin__open":     {"keyword": "open",     "forbidden_nodes": [],  "forbidden_names": ["open"]},
    "builtin__type":     {"keyword": "type",     "forbidden_nodes": [],  "forbidden_names": ["type"]},
    "builtin__hash":     {"keyword": "hash",     "forbidden_nodes": [],  "forbidden_names": ["hash"]},
    "builtin__id":       {"keyword": "id",       "forbidden_nodes": [],  "forbidden_names": ["id"]},
    "builtin__all":      {"keyword": "all",      "forbidden_nodes": [],  "forbidden_names": ["all"]},
    "builtin__any":      {"keyword": "any",      "forbidden_nodes": [],  "forbidden_names": ["any"]},
    "builtin__sum":      {"keyword": "sum",      "forbidden_nodes": [],  "forbidden_names": ["sum"]},
    "builtin__min":      {"keyword": "min",      "forbidden_nodes": [],  "forbidden_names": ["min"]},
    "builtin__max":      {"keyword": "max",      "forbidden_nodes": [],  "forbidden_names": ["max"]},
    "builtin__next":     {"keyword": "next",     "forbidden_nodes": [],  "forbidden_names": ["next"]},
    "builtin__input":    {"keyword": "input",    "forbidden_nodes": [],  "forbidden_names": ["input"]},
    "builtin__len":      {"keyword": "len",      "forbidden_nodes": [],  "forbidden_names": ["len"]},
    "builtin__range":    {"keyword": "range",    "forbidden_nodes": [],  "forbidden_names": ["range"]},
    "builtin__filter":   {"keyword": "filter",   "forbidden_nodes": [],  "forbidden_names": ["filter"]},
    "builtin__print":    {"keyword": "print",    "forbidden_nodes": [],  "forbidden_names": ["print"]},
    "builtin__int":      {"keyword": "int",      "forbidden_nodes": [],  "forbidden_names": ["int"]},
    "builtin__float":    {"keyword": "float",    "forbidden_nodes": [],  "forbidden_names": ["float"]},
    "builtin__str":      {"keyword": "str",      "forbidden_nodes": [],  "forbidden_names": ["str"]},
    "builtin__bool":     {"keyword": "bool",     "forbidden_nodes": [],  "forbidden_names": ["bool"]},
    "builtin__round":    {"keyword": "round",    "forbidden_nodes": [],  "forbidden_names": ["round"]},
    "builtin__zip":      {"keyword": "zip",      "forbidden_nodes": [],  "forbidden_names": ["zip"]},
    "builtin__sorted":   {"keyword": "sorted",   "forbidden_nodes": [],  "forbidden_names": ["sorted"]},
    "builtin__super":    {"keyword": "super",    "forbidden_nodes": [],  "forbidden_names": ["super"]},
    "builtin__iter":     {"keyword": "iter",     "forbidden_nodes": [],  "forbidden_names": ["iter"]},
    "builtin__object":   {"keyword": "object",   "forbidden_nodes": [],  "forbidden_names": ["object"]},
    "builtin__bytes":    {"keyword": "bytes",    "forbidden_nodes": [],  "forbidden_names": ["bytes"]},
    "builtin__dict":     {"keyword": "dict",     "forbidden_nodes": [],  "forbidden_names": ["dict"]},
    "builtin__tuple":    {"keyword": "tuple",    "forbidden_nodes": [],  "forbidden_names": ["tuple"]},
    "builtin__property": {"keyword": "property", "forbidden_nodes": [],  "forbidden_names": ["property"]},
    "builtin__complex":  {"keyword": "complex",  "forbidden_nodes": [],  "forbidden_names": ["complex"]},
    "builtin__reversed": {"keyword": "reversed", "forbidden_nodes": [],  "forbidden_names": ["reversed"]},
}
print(f"KEYWORD_MAP: {len(KEYWORD_MAP)} objects")

In [ ]:
# Cell 3 – Load functions

def load_universal_masks(eps, cons):
    """Load universal object masks from 13_object_masks file."""
    path = os.path.join(DATA_DIR, f"13_object_masks_eps{eps}_cons{cons}.h5")
    masks = {}
    with h5py.File(path, "r") as f:
        um = f["universal_masks"]
        for lid in range(N_LAYERS):
            lk = f"layer_{lid}"
            if lk not in um:
                continue
            for ds_name in um[lk]:
                if ds_name not in masks:
                    masks[ds_name] = {}
                masks[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
    return masks

def load_checker_masks(eps, cons):
    """Load checker masks from 13_checker_masks file."""
    path = os.path.join(DATA_DIR, f"13_checker_masks_eps{eps}_cons{cons}.h5")
    masks = {}
    with h5py.File(path, "r") as f:
        tcm = f["token_checker_masks"]
        for lid in range(N_LAYERS):
            lk = f"layer_{lid}"
            if lk not in tcm:
                continue
            for ds_name in tcm[lk]:
                if ds_name not in masks:
                    masks[ds_name] = {}
                masks[ds_name][lid] = np.array(tcm[lk][ds_name], dtype=bool)
    return masks

print("Load functions defined.")

In [ ]:
# Cell 4 – Compare function

def compare_masks(universal_mask, token_mask):
    A = universal_mask
    B = token_mask
    a_only = int(np.logical_and(A, ~B).sum())
    a_and_b = int(np.logical_and(A, B).sum())
    b_only = int(np.logical_and(~A, B).sum())
    a_size = int(A.sum())
    b_size = int(B.sum())
    token_fraction = float(a_and_b / a_size) if a_size > 0 else 0.0
    concept_fraction = float(a_only / a_size) if a_size > 0 else 0.0
    return {
        "A_size": a_size, "B_size": b_size,
        "A_only": a_only, "A_and_B": a_and_b, "B_only": b_only,
        "token_fraction": round(token_fraction, 4),
        "concept_fraction": round(concept_fraction, 4),
    }

print("Compare function defined.")

In [ ]:
# Cell 5 – Main comparison loop
from tqdm.auto import tqdm

all_summary_rows = []

for eps in EPSILONS:
    for cons in CONSISTENCIES:
        print(f"\n--- eps={eps}, cons={cons} ---")
        universal = load_universal_masks(eps, cons)
        checker = load_checker_masks(eps, cons)
        
        testable = sorted(set(universal.keys()) & set(checker.keys()) & set(KEYWORD_MAP.keys()))
        print(f"  Testable objects: {len(testable)}")
        
        if len(testable) == 0:
            print("  SKIPPED: no overlapping objects")
            continue
        
        # Build per-layer comparison
        rows = []
        for obj in testable:
            for lid in range(N_LAYERS):
                A = universal[obj].get(lid, np.zeros(2048, dtype=bool))
                B = checker[obj].get(lid, np.zeros(2048, dtype=bool))
                result = compare_masks(A, B)
                rows.append({
                    "target": obj,
                    "layer": lid,
                    **result,
                })
        
        comp_df = pd.DataFrame(rows)
        
        # Save target_checker CSV (same format as target_checker_1.csv)
        comp_df["entry"] = comp_df.apply(
            lambda r: f"{r['A_only']} / {r['A_and_B']} / {r['B_only']}", axis=1
        )
        pivot = comp_df.pivot(index="target", columns="layer", values="entry")
        pivot.columns = [f"L{c}" for c in pivot.columns]
        
        csv_name = f"14_target_checker_eps{eps}_cons{cons}.csv"
        csv_path = os.path.join(DATA_DIR, csv_name)
        pivot.to_csv(csv_path)
        print(f"  Saved: {csv_name}")
        
        # Summary stats for this combo
        n_ast = sum(1 for o in testable if o.startswith("ast__"))
        n_blt = sum(1 for o in testable if o.startswith("builtin__"))
        
        ast_subset = comp_df[comp_df["target"].str.startswith("ast__")]
        blt_subset = comp_df[comp_df["target"].str.startswith("builtin__")]
        
        all_summary_rows.append({
            "epsilon": eps,
            "consistency": cons,
            "n_testable": len(testable),
            "n_ast": n_ast,
            "n_builtin": n_blt,
            "mean_concept_fraction": round(comp_df["concept_fraction"].mean(), 4),
            "mean_token_fraction": round(comp_df["token_fraction"].mean(), 4),
            "ast_concept_fraction": round(ast_subset["concept_fraction"].mean(), 4) if len(ast_subset) > 0 else None,
            "ast_token_fraction": round(ast_subset["token_fraction"].mean(), 4) if len(ast_subset) > 0 else None,
            "blt_concept_fraction": round(blt_subset["concept_fraction"].mean(), 4) if len(blt_subset) > 0 else None,
            "blt_token_fraction": round(blt_subset["token_fraction"].mean(), 4) if len(blt_subset) > 0 else None,
            "mean_A_size": round(comp_df["A_size"].mean(), 1),
            "mean_B_size": round(comp_df["B_size"].mean(), 1),
        })

In [ ]:
# Cell 6 – Summary table
summary_df = pd.DataFrame(all_summary_rows)
summary_path = os.path.join(DATA_DIR, "14_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("=== THRESHOLD SWEEP SUMMARY ===\n")
display(summary_df)
print(f"\nSaved: {summary_path}")

In [ ]:
# Cell 7 – Key observations

print("=== KEY OBSERVATIONS ===\n")

# How does concept_fraction change with thresholds?
print("Concept fraction by threshold (higher = more concept-driven):")
for _, row in summary_df.iterrows():
    print(f"  eps={row['epsilon']:<6} cons={row['consistency']:<4}  "
          f"concept={row['mean_concept_fraction']:.3f}  "
          f"token={row['mean_token_fraction']:.3f}  "
          f"AST_concept={row['ast_concept_fraction']}  "
          f"BLT_concept={row['blt_concept_fraction']}")

print()

# Do builtins become more token-driven at lower thresholds?
if len(summary_df) > 1:
    strict = summary_df[(summary_df["epsilon"] == 0.001) & (summary_df["consistency"] == 0.8)]
    permissive = summary_df[(summary_df["epsilon"] == 0.001) & (summary_df["consistency"] == 0.2)]
    if len(strict) > 0 and len(permissive) > 0:
        s = strict.iloc[0]
        p = permissive.iloc[0]
        print(f"Effect of lowering consistency (eps=0.001):")
        print(f"  cons=0.8: concept={s['mean_concept_fraction']:.3f}, token={s['mean_token_fraction']:.3f}")
        print(f"  cons=0.2: concept={p['mean_concept_fraction']:.3f}, token={p['mean_token_fraction']:.3f}")
        if p['blt_concept_fraction'] is not None and s['blt_concept_fraction'] is not None:
            print(f"  Builtin concept fraction: {s['blt_concept_fraction']:.3f} -> {p['blt_concept_fraction']:.3f}")